# Kaggle → ADNI Transfer Learning

## Setup

In [ ]:
# ── KAGGLE SETUP: Download ADNI dataset from Google Drive ───────────────

import os, subprocess
from pathlib import Path

GDRIVE_FILE_ID = '1uc4Nb_z6_q5DzqYfN9EtRagtstkPf2rz'
WORK_DIR = Path('/kaggle/working')
ADNI_ZIP_PATH = WORK_DIR / 'adni_dataset.zip'

if not os.path.exists(WORK_DIR / 'CN'):
    print('📦 Step 1/3: Installing gdown...')
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    
    print('⬇️  Step 2/3: Downloading ADNI dataset from Google Drive...')
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}', str(ADNI_ZIP_PATH), quiet=False)
    
    print('📂 Step 3/3: Extracting ADNI Dataset...')
    import zipfile
    with zipfile.ZipFile(str(ADNI_ZIP_PATH), 'r') as zf:
        zf.extractall(str(WORK_DIR))
    ADNI_ZIP_PATH.unlink()
    print('✅ Done! ADNI Dataset ready.')

ADNI_DIR = str(WORK_DIR)
for root, dirs, files in os.walk(WORK_DIR):
    if 'CN' in dirs and 'MCI' in dirs and 'AD' in dirs:
        ADNI_DIR = root
        break

KAGGLE_DIR = '/kaggle/input'
for root, dirs, files in os.walk('/kaggle/input'):
    if 'MildDemented' in dirs and 'ModerateDemented' in dirs:
        KAGGLE_DIR = root
        break

print(f"\nADNI_DIR = {ADNI_DIR}")
print(f"KAGGLE_DIR = {KAGGLE_DIR}")

In [ ]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Data Mapping & Subject-Level Splitting
Kaggle has 4 classes, ADNI has 3. We use your mapping:
* Kaggle `NonDemented` -> ADNI `CN`
* Kaggle `VeryMildDemented` -> ADNI `MCI`
* Kaggle `MildDemented` & `ModerateDemented` -> ADNI `AD`


In [ ]:
class_mapping = {
    'NonDemented': 0,
    'VeryMildDemented': 1,
    'MildDemented': 2,
    'ModerateDemented': 2,
    
    'CN': 0,
    'MCI': 1,
    'AD': 2
}

### Subject-Level Splitting for ADNI
To prevent data leakage, we split by **Subject ID**, not by individual images.


In [ ]:
def get_adni_subject_splits(adni_dir, train_ratio=0.7, val_ratio=0.15):
    subjects = []
    labels = []
    
    for class_name in ['CN', 'MCI', 'AD']:
        class_dir = os.path.join(adni_dir, class_name)
        if not os.path.exists(class_dir): continue
            
        for subj_id in os.listdir(class_dir):
            if os.path.isdir(os.path.join(class_dir, subj_id)):
                subjects.append((subj_id, class_name))
                labels.append(class_name)
                
    from collections import defaultdict
    class_subjs = defaultdict(list)
    for s, l in subjects:
        class_subjs[l].append(s)
        
    train_subjs, val_subjs, test_subjs = set(), set(), set()
    for c, subjs in class_subjs.items():
        random.shuffle(subjs)
        n = len(subjs)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        
        train_subjs.update(subjs[:n_train])
        val_subjs.update(subjs[n_train:n_train+n_val])
        test_subjs.update(subjs[n_train+n_val:])
        
    return train_subjs, val_subjs, test_subjs

train_subjs, val_subjs, test_subjs = get_adni_subject_splits(ADNI_DIR)
print(f"ADNI Subjects - Train: {len(train_subjs)}, Val: {len(val_subjs)}, Test: {len(test_subjs)}")

### Custom Dataset Class


In [ ]:
class AlzheimerDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform
        
    def __len__(self):
        return len(self.data_list)
        
    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
kaggle_data = []
for c in ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']:
    c_dir = os.path.join(KAGGLE_DIR, c)
    if os.path.exists(c_dir):
        for img in os.listdir(c_dir):
            if img.endswith('.jpg') or img.endswith('.png'):
                kaggle_data.append((os.path.join(c_dir, img), class_mapping[c]))
                
random.shuffle(kaggle_data)
split_idx = int(len(kaggle_data) * 0.8)
kaggle_train = AlzheimerDataset(kaggle_data[:split_idx], transform=train_transform)
kaggle_val = AlzheimerDataset(kaggle_data[split_idx:], transform=val_transform)

kaggle_train_loader = DataLoader(kaggle_train, batch_size=32, shuffle=True)
kaggle_val_loader = DataLoader(kaggle_val, batch_size=32, shuffle=False)

## 2. Phase 1: Pre-training on Kaggle


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, save_path='model.pth'):
    best_acc = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss, running_corrects = 0.0, 0
        
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} Train'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                
        val_acc = val_corrects.double() / len(val_loader.dataset)
        print(f'Epoch {epoch+1} Val Acc: {val_acc:.4f}')
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            
    return model

model = train_model(model, kaggle_train_loader, kaggle_val_loader, criterion, optimizer, num_epochs=15, save_path='kaggle_pretrained.pth')

## 3. Phase 2: Fine-tuning on ADNI (Domain Adaptation)


In [ ]:
adni_train_data, adni_val_data, adni_test_data = [], [], []

for c in ['CN', 'MCI', 'AD']:
    c_dir = os.path.join(ADNI_DIR, c)
    if not os.path.exists(c_dir): continue
    for subj_id in os.listdir(c_dir):
        subj_dir = os.path.join(c_dir, subj_id)
        if not os.path.isdir(subj_dir): continue
            
        imgs = [os.path.join(subj_dir, img) for img in os.listdir(subj_dir) if img.endswith('.png')]
        for img in imgs:
            item = (img, class_mapping[c])
            if subj_id in train_subjs:
                adni_train_data.append(item)
            elif subj_id in val_subjs:
                adni_val_data.append(item)
            elif subj_id in test_subjs:
                adni_test_data.append(item)
                
adni_train_loader = DataLoader(AlzheimerDataset(adni_train_data, train_transform), batch_size=32, shuffle=True)
adni_val_loader = DataLoader(AlzheimerDataset(adni_val_data, val_transform), batch_size=32, shuffle=False)
adni_test_loader = DataLoader(AlzheimerDataset(adni_test_data, val_transform), batch_size=32, shuffle=False)

print(f"ADNI Images - Train: {len(adni_train_data)}, Val: {len(adni_val_data)}, Test: {len(adni_test_data)}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

adni_train_labels = [label for _, label in adni_train_data]
classes = np.unique(adni_train_labels)
class_weights = compute_class_weight('balanced', classes=classes, y=adni_train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("ADNI Class Weights [CN, MCI, AD]:", class_weights)

criterion_ft = nn.CrossEntropyLoss(weight=class_weights_tensor)

In [ ]:
model.load_state_dict(torch.load('kaggle_pretrained.pth'))

optimizer_ft = optim.Adam(model.parameters(), lr=1e-5) 

model = train_model(model, adni_train_loader, adni_val_loader, criterion_ft, optimizer_ft, num_epochs=10, save_path='adni_finetuned.pth')

## 4. Final Evaluation (Subject-Level Testing)
We evaluate ONLY on ADNI patients the model has never seen.


In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['CN', 'MCI', 'AD']))
    
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['CN', 'MCI', 'AD'], yticklabels=['CN', 'MCI', 'AD'])
    plt.title("Subject-Level Confusion Matrix (ADNI)")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

model.load_state_dict(torch.load('adni_finetuned.pth'))
evaluate_model(model, adni_test_loader)

## 5. Explainable AI (XAI) on the fine-tuned ResNet-50

We probe the fine-tuned ResNet50 with four complementary explainability techniques:
1. **Grad-CAM** — class-discriminative localization from the last conv block.
2. **LIME** — model-agnostic local surrogate over super-pixel perturbations.
3. **SHAP** (GradientExplainer) — Shapley-value attributions over pixels.
4. **Integrated Gradients** — path-integrated attributions from a black baseline.

In [ ]:
!pip install grad-cam lime shap -q

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shap
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from lime import lime_image
from skimage.segmentation import mark_boundaries

XAI_CLASS_NAMES = ['CN', 'MCI', 'AD']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _denorm(t):
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    arr = t.detach().cpu().numpy()
    if arr.ndim == 4: arr = arr[0]
    return np.clip(np.transpose(arr * std + mean, (1, 2, 0)), 0, 1)

def pick_one_per_class(test_loader, class_names, device):
    samples = {}
    model.eval()
    for images, labels in test_loader:
        for img, lbl in zip(images, labels):
            lbl = int(lbl)
            if lbl not in samples:
                samples[lbl] = img.unsqueeze(0).to(device)
            if len(samples) == len(class_names):
                break
        if len(samples) == len(class_names):
            break
    return samples

xai_samples = pick_one_per_class(adni_test_loader, XAI_CLASS_NAMES, device)
print('XAI samples ready for classes:',
      [XAI_CLASS_NAMES[k] for k in sorted(xai_samples)])


### 5.1 Grad-CAM

In [ ]:
target_layer = model.layer4[-1]
cam = GradCAM(model=model, target_layers=[target_layer])
for cls_idx, img_tensor in sorted(xai_samples.items()):
    cls_name = XAI_CLASS_NAMES[cls_idx]
    grayscale = cam(input_tensor=img_tensor,
                    targets=[ClassifierOutputTarget(cls_idx)])[0]
    img_rgb = _denorm(img_tensor)
    overlay_heat = cv2.applyColorMap((grayscale * 255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay_heat = cv2.cvtColor(overlay_heat, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.5 * img_rgb + 0.5 * overlay_heat
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(img_rgb); ax[0].set_title(f'Input ({cls_name})'); ax[0].axis('off')
    ax[1].imshow(grayscale, cmap='hot'); ax[1].set_title('Grad-CAM'); ax[1].axis('off')
    ax[2].imshow(overlay); ax[2].set_title('Overlay'); ax[2].axis('off')
    plt.suptitle(f'Grad-CAM — {cls_name}'); plt.tight_layout(); plt.show()


### 5.2 LIME

In [ ]:
def lime_predict_fn(images_np):
    imgs = torch.from_numpy(images_np).float().permute(0, 3, 1, 2).to(device)
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1)
    imgs = (imgs - mean) / std
    with torch.no_grad():
        return torch.softmax(model(imgs), dim=1).cpu().numpy()

explainer = lime_image.LimeImageExplainer()
for cls_idx, img_tensor in sorted(xai_samples.items()):
    cls_name = XAI_CLASS_NAMES[cls_idx]
    img_rgb = _denorm(img_tensor)
    explanation = explainer.explain_instance(
        image=img_rgb, classifier_fn=lime_predict_fn,
        labels=[cls_idx], num_samples=500
    )
    img, mask = explanation.get_image_and_mask(
        label=cls_idx, positive_only=False, num_features=8, hide_rest=False
    )
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(img_rgb); ax[0].set_title(f'Input ({cls_name})'); ax[0].axis('off')
    ax[1].imshow(mark_boundaries(img, mask)); ax[1].set_title('LIME explanation'); ax[1].axis('off')
    plt.suptitle(f'LIME — {cls_name}'); plt.tight_layout(); plt.show()


### 5.3 SHAP (GradientExplainer)

In [ ]:
bg_images = []
for images, _ in adni_test_loader:
    bg_images.append(images)
    if sum(b.size(0) for b in bg_images) >= 16: break
bg = torch.cat(bg_images, dim=0)[:16].to(device)

shap_explainer = shap.GradientExplainer(model, bg)
for cls_idx, img_tensor in sorted(xai_samples.items()):
    cls_name = XAI_CLASS_NAMES[cls_idx]
    sv = shap_explainer.shap_values(img_tensor)
    smap = np.asarray(sv[cls_idx][0] if isinstance(sv, list) else sv[0])
    if smap.ndim == 4: smap = smap[0]
    imp = smap.sum(axis=0)
    img_rgb = _denorm(img_tensor)
    H, W = img_rgb.shape[:2]
    imp = cv2.resize(imp.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    vmax = np.abs(imp).max() + 1e-8
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(img_rgb); ax[0].set_title(f'Input ({cls_name})'); ax[0].axis('off')
    ax[1].imshow(img_rgb)
    over = ax[1].imshow(imp, cmap='coolwarm', alpha=0.5, vmin=-vmax, vmax=vmax)
    ax[1].set_title('SHAP attributions'); ax[1].axis('off')
    plt.colorbar(over, ax=ax[1], fraction=0.046, pad=0.04)
    plt.suptitle(f'SHAP — {cls_name}'); plt.tight_layout(); plt.show()


### 5.4 Integrated Gradients

In [ ]:
def integrated_gradients(model_, img_tensor, target_class, steps=50):
    model_.eval()
    x = img_tensor.to(device)
    baseline = torch.zeros_like(x)
    grads = torch.zeros_like(x)
    for a in torch.linspace(0.0, 1.0, steps).to(device):
        xi = (baseline + a * (x - baseline)).detach().requires_grad_(True)
        logits = model_(xi)
        score = logits[0, target_class]
        g = torch.autograd.grad(score, xi)[0]
        grads = grads + g.detach()
    ig = (x - baseline) * (grads / steps)
    return ig.detach().cpu().numpy()[0]

for cls_idx, img_tensor in sorted(xai_samples.items()):
    cls_name = XAI_CLASS_NAMES[cls_idx]
    ig_map = integrated_gradients(model, img_tensor, cls_idx, steps=50)
    attr = ig_map.sum(axis=0)
    img_rgb = _denorm(img_tensor)
    vmax = np.abs(attr).max() + 1e-8
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(img_rgb); ax[0].set_title(f'Input ({cls_name})'); ax[0].axis('off')
    im = ax[1].imshow(attr, cmap='seismic', vmin=-vmax, vmax=vmax)
    ax[1].set_title('IG attributions'); ax[1].axis('off')
    plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)
    ax[2].imshow(img_rgb)
    ax[2].imshow(attr, cmap='seismic', vmin=-vmax, vmax=vmax, alpha=0.5)
    ax[2].set_title('Overlay'); ax[2].axis('off')
    plt.suptitle(f'Integrated Gradients — {cls_name}'); plt.tight_layout(); plt.show()
